In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import joblib

def export_isolation_forest_to_cpp(model, filename="IsolationForest.h"):
    """
    Custom exporter that extracts every tree from an sklearn IsolationForest
    and formats them as raw C++ arrays for an ESP32.
    """
    with open(filename, "w") as f:
        f.write("// Auto-generated Isolation Forest model for ESP32\n")
        f.write("#ifndef ISOLATION_FOREST_H\n")
        f.write("#define ISOLATION_FOREST_H\n\n")

        n_trees = len(model.estimators_)
        f.write(f"const int N_TREES = {n_trees};\n\n")

        # 1. Extract and write the arrays for every single tree
        for i, tree_model in enumerate(model.estimators_):
            t = tree_model.tree_
            f.write(f"const int tree_{i}_feature[] = {{{', '.join(map(str, t.feature))}}};\n")
            f.write(f"const float tree_{i}_threshold[] = {{{', '.join(map(str, t.threshold))}}};\n")
            f.write(f"const int tree_{i}_left[] = {{{', '.join(map(str, t.children_left))}}};\n")
            f.write(f"const int tree_{i}_right[] = {{{', '.join(map(str, t.children_right))}}};\n\n")

        # 2. Create arrays of pointers to easily loop through all trees in C++
        f.write("// Arrays of pointers to loop through the trees\n")
        f.write("const int* const features[] = {" + ', '.join([f"tree_{i}_feature" for i in range(n_trees)]) + "};\n")
        f.write("const float* const thresholds[] = {" + ', '.join([f"tree_{i}_threshold" for i in range(n_trees)]) + "};\n")
        f.write("const int* const lefts[] = {" + ', '.join([f"tree_{i}_left" for i in range(n_trees)]) + "};\n")
        f.write("const int* const rights[] = {" + ', '.join([f"tree_{i}_right" for i in range(n_trees)]) + "};\n\n")

        # 3. Write the C++ traversal functions
        f.write("""
// Traverses a single tree and returns the path length (depth)
float traverse_tree(int tree_idx, const float x[]) {
    int node = 0;
    float depth = 0.0;

    // A left child of -1 indicates a leaf node in scikit-learn
    while (lefts[tree_idx][node] != -1) {
        if (x[features[tree_idx][node]] <= thresholds[tree_idx][node]) {
            node = lefts[tree_idx][node];
        } else {
            node = rights[tree_idx][node];
        }
        depth += 1.0;
    }
    return depth;
}

// Computes the average path length across all trees
// In an Isolation Forest, a LOWER average depth means a HIGHER chance of anomaly!
float get_average_depth(const float x[]) {
    float avg_depth = 0.0;
    for (int i = 0; i < N_TREES; i++) {
        avg_depth += traverse_tree(i, x);
    }
    avg_depth /= N_TREES;
    return avg_depth;
}
""")
        f.write("\n#endif\n")


# 1. Load the master dataset you exported from KNIME
print("Loading data...")
df = pd.read_csv("CIVILEA_ESP32_Master_Training_Data.csv")

# 2. Define our input features (what the AI sees) and our Answer Key (what the AI tries to predict)
features = ["TDS_V", "Flow_Lmin", "Distance_cm", "Rain_ADC", "Soil_ADC"]
X = df[features]
y_true = df["Overflow_Switch"] # Our physical answer key

# 3. Split the data! 80% for training, 20% for the final exam
print("Splitting data into 80% Train and 20% Test...")
X_train, X_test, y_train, y_test = train_test_split(X, y_true, test_size=0.20, random_state=42)

# 4. Instantiate the Isolation Forest
print("Training the Isolation Forest on the 80% dataset...")
model = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

# 5. TRAIN the model strictly on X_train
model.fit(X_train)

# 6. TEST the model on the unseen X_test
print("Testing the model on the 20% unseen data...")
predictions = model.predict(X_test)
y_pred = [1 if pred == -1 else 0 for pred in predictions]

print("\n--- FINAL TEST RESULTS ---")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Normal (0)", "Anomaly/Flood (1)"]))

# 7. Save standard PC model
joblib.dump(model, "civilea_esp32_isolation_forest.pkl")

# 8. EXPORT TO CUSTOM C++ HEADER
print("\nRunning custom C++ exporter...")
export_isolation_forest_to_cpp(model, "IsolationForest.h")
print("SUCCESS: Arrays and traversal logic saved to 'IsolationForest.h'!")

Loading data...
Splitting data into 80% Train and 20% Test...
Training the Isolation Forest on the 80% dataset...
Testing the model on the 20% unseen data...

--- FINAL TEST RESULTS ---
[[56438     1]
 [    0    59]]
                   precision    recall  f1-score   support

       Normal (0)       1.00      1.00      1.00     56439
Anomaly/Flood (1)       0.98      1.00      0.99        59

         accuracy                           1.00     56498
        macro avg       0.99      1.00      1.00     56498
     weighted avg       1.00      1.00      1.00     56498


Running custom C++ exporter...
SUCCESS: Arrays and traversal logic saved to 'IsolationForest.h'!
